# 🎬 Movie Recommender System
**Content-Based Filtering using TMDB 5000 Dataset**

### Setup
1. Download datasets from Kaggle: [TMDB 5000 Movie Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata)
2. Place `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv` in the same folder as this notebook
3. Run all cells top to bottom

### Compatible with
- Python 3.10 → 3.14
- Jupyter Notebook / JupyterLab (latest)
- PyCharm 2026.1
- scikit-learn ≥ 1.3, pandas ≥ 2.0, numpy ≥ 1.26

## Step 1 — Imports

In [17]:
import ast
import pickle
import os

import numpy as np
import pandas as pd
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.porter import PorterStemmer

nltk.download('punkt', quiet=True)
ps = PorterStemmer()

print('All imports OK')

All imports OK


## Step 2 — Load & Merge Datasets

In [18]:
movies  = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

print('Movies shape :', movies.shape)
print('Credits shape:', credits.shape)

Movies shape : (4803, 20)
Credits shape: (4803, 4)


In [19]:
movies = movies.merge(credits, on='title')
print('Merged shape:', movies.shape)

Merged shape: (4809, 23)


In [20]:
# Keep only the columns we need
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


## Step 3 — Clean Data

In [21]:
print('Null values before cleaning:')
print(movies.isnull().sum())

Null values before cleaning:
movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64


In [6]:
movies.dropna(inplace=True)
movies.drop_duplicates(inplace=True)
movies = movies.reset_index(drop=True)   # ← critical: keeps iloc and label index aligned
print('Clean shape:', movies.shape)

Clean shape: (4806, 7)


## Step 4 — Feature Engineering

In [22]:
def extract_names(text: str) -> list[str]:
    """Parse JSON-like string and return list of 'name' values."""
    try:
        return [item['name'] for item in ast.literal_eval(text)]
    except (ValueError, SyntaxError):
        return []


def extract_top3_cast(text: str) -> list[str]:
    """Return first 3 cast member names."""
    try:
        return [item['name'] for item in ast.literal_eval(text)[:3]]
    except (ValueError, SyntaxError):
        return []


def extract_director(text: str) -> list[str]:
    """Return director name(s) from crew JSON string."""
    try:
        return [item['name'] for item in ast.literal_eval(text)
                if item.get('job') == 'Director']
    except (ValueError, SyntaxError):
        return []


def collapse_spaces(tokens: list[str]) -> list[str]:
    """Remove spaces inside multi-word tokens (e.g. 'Sam Worthington' -> 'SamWorthington')."""
    return [t.replace(' ', '') for t in tokens]


print('Helper functions defined')

Helper functions defined


In [8]:
movies['genres']   = movies['genres'].apply(extract_names).apply(collapse_spaces)
movies['keywords'] = movies['keywords'].apply(extract_names).apply(collapse_spaces)
movies['cast']     = movies['cast'].apply(extract_top3_cast).apply(collapse_spaces)
movies['crew']     = movies['crew'].apply(extract_director).apply(collapse_spaces)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


In [9]:
# Combine all features into one 'tags' column
movies['tags'] = (
    movies['overview'] +
    movies['genres']   +
    movies['keywords'] +
    movies['cast']     +
    movies['crew']
)

# Drop intermediate columns; keep movie_id, title, tags
new_df = movies[['movie_id', 'title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda tokens: ' '.join(tokens)).str.lower()
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


## Step 5 — Stemming

Porter Stemmer reduces words to their root so *'action'*, *'actions'*, *'acting'* all match.

In [23]:
def stem(text: str) -> str:
    return ' '.join(ps.stem(word) for word in text.split())


new_df['tags'] = new_df['tags'].apply(stem)
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."


## Step 6 — TF-IDF Vectorization

TF-IDF down-weights common words (e.g. *'the'*, *'love'*) and boosts
distinctive words — giving sharper cosine-similarity separation than raw CountVectorizer.

In [24]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', sublinear_tf=True)
vectors = tfidf.fit_transform(new_df['tags']).toarray()

print('Vector shape:', vectors.shape)

Vector shape: (4806, 5000)


## Step 7 — Cosine Similarity Matrix

In [27]:
similarity = cosine_similarity(vectors)
print('Similarity matrix shape:', similarity.shape)

Similarity matrix shape: (4806, 4806)


## Step 8 — Recommend Function

In [25]:
def recommend(movie: str, df: pd.DataFrame, sim_matrix: np.ndarray, n: int = 5) -> list[str]:
    """
    Return top-n similar movie titles for a given movie.
    Uses label==positional index (guaranteed by reset_index in Step 3).
    """
    matches = df[df['title'] == movie]
    if matches.empty:
        return [f'Movie "{movie}" not found in dataset']

    idx = matches.index[0]                                         # positional after reset
    distances = sorted(enumerate(sim_matrix[idx]), key=lambda x: x[1], reverse=True)
    return [df.iloc[i[0]]['title'] for i in distances[1: n + 1]]


# Quick test
for title in recommend('Batman Begins', new_df, similarity):
    print(title)

The Dark Knight
The Dark Knight Rises
Batman v Superman: Dawn of Justice
Batman
Batman Returns


## Step 9 — Save Artifacts

In [26]:
os.makedirs('model', exist_ok=True)

with open('model/movie_list.pkl', 'wb') as f:
    pickle.dump(new_df, f)

with open('model/similarity.pkl', 'wb') as f:
    pickle.dump(similarity, f)

print('Saved  model/movie_list.pkl')
print('Saved  model/similarity.pkl')

Saved  model/movie_list.pkl
Saved  model/similarity.pkl
